# CLIP Text and Image Embedding Demo

This notebook demonstrates the core idea behind multimodal embeddings. It encodes a text prompt and an image with CLIP, normalizes both vectors, and calculates their similarity score.


## 1. Import Required Libraries

The notebook uses Hugging Face Transformers for CLIP, Pillow for image loading, and PyTorch for tensor operations.


In [1]:
from transformers import CLIPProcessor , CLIPModel
from PIL import Image
import torch 

/home/lakshay/Multimodal_Embeddings/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load the CLIP Model and Processor

The CLIP model maps text and images into a shared semantic space. The processor converts raw text and images into model-ready tensors.


In [2]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 60402.11it/s]


## 3. Generate a Text Embedding

The text prompt is tokenized, passed through CLIP, and converted into a fixed-size feature vector. The printed shape confirms the embedding dimensions.


In [7]:
text = ['a dog running']

input = processor(
    text=text,
    return_tensors='pt',
    padding=True
)

with torch.no_grad():
    text_output = model.get_text_features(**input)

# Handle both tensor output and model output objects.
if isinstance(text_output, torch.Tensor):
    text_features = text_output
elif hasattr(text_output, "pooler_output") and text_output.pooler_output is not None:
    text_features = text_output.pooler_output
elif hasattr(text_output, "last_hidden_state"):
    text_features = text_output.last_hidden_state
else:
    raise TypeError(f"Unexpected output type from get_text_features: {type(text_output)}")

print(text_features.shape)

torch.Size([1, 512])


## 4. Generate an Image Embedding

The image is loaded, preprocessed, and encoded into the same semantic space as the text prompt. This allows direct comparison between text and image features.


In [10]:
image = Image.open("/home/lakshay/Multimodal_Embeddings/image.png")

inputs = processor(
    images=image,
    return_tensors="pt"
)

with torch.no_grad():
    image_output = model.get_image_features(**inputs)

# Handle both tensor output and model output objects.
if isinstance(image_output, torch.Tensor):
    image_features = image_output
elif hasattr(image_output, "pooler_output") and image_output.pooler_output is not None:
    image_features = image_output.pooler_output
elif hasattr(image_output, "last_hidden_state"):
    image_features = image_output.last_hidden_state
else:
    raise TypeError(f"Unexpected output type from get_image_features: {type(image_output)}")

print(image_features.shape)

torch.Size([1, 512])


## 5. Normalize the Embeddings

Both vectors are L2-normalized so the dot product measures directional similarity rather than raw vector magnitude.


In [11]:
text_features = text_features / text_features.norm(
    dim=-1,
    keepdim=True
)

image_features = image_features / image_features.norm(
    dim=-1,
    keepdim=True
)

## 6. Calculate Text-Image Similarity

The final score is the dot product between the normalized text and image embeddings. Higher values indicate stronger semantic alignment between the prompt and the image.


In [12]:
similarity = torch.matmul(
    text_features,
    image_features.T
)

print(similarity.item())

0.305262953042984
